### EY AI & Data Challenge Program Model

In [1]:
import pandas as pd
from typing import Union, Callable
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

##### Read and Preprocess

- The data from "data/good_vs_bad.pkl" has the our final dataset including added feature engineering columns for ideal/unacceptable water quality.
- We can use one hot encoding to convert columns with dates and strings to integers and floats
- Lastly, take the Series that are already int/float dtypes and concatenate them with our new onehot encoded data, preparing for the train-test split

In [2]:
def get_wq_data(file: str = "data/good_vs_bad.pkl") -> pd.DataFrame:
    """read latest file with as many attributes as possible
    input
    -----
    file: str path

    return
    ------
    pd.DataFrame of dataset ready for model
    """

    try:
        df = pd.read_pickle(file)

    except:
        df = pd.read_csv(file)
    pd.set_option('future.no_silent_downcasting', True)

    # specify month for each sample
    try:
        df['month'] = pd.to_datetime(df['month']).apply(lambda d: d.strftime("%Y-%m"))
    except:
        pass
    
    # apply get_dummies function
    df_encoded = pd.get_dummies(df[['province', 'country', 'sample date', 'month', 'sample_year']], dtype=int)
    df_encoded.head()
    
    # One hot encoded data, plus original data already float or int
    df = pd.concat([df.select_dtypes(include=["float64", "int64"]), df_encoded], axis=1)
 
    return df

##### Train-Test Split

- Split the data into training/test data

In [3]:
def std_train_test_split(
    df: pd.DataFrame, *, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Returns train-test split"""
 
    feature_df = df.drop(columns=['total alkalinity', 'electrical conductance', 'dissolved reactive phosphorus'])
    X = df[list(feature_df.columns)]
    y = df[['total alkalinity', 'electrical conductance', 'dissolved reactive phosphorus']].astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    # raise NotImplementedError()
 
    return X_train, X_test, y_train, y_test

#### Use Scikit Learn's DecisionTreeRegressor

- First, run the base model on our initial dataset ```"wq.csv"```.

In [4]:
def baseline_model(
    tts: Callable, *, test_size: Union[float, int] = 0.2, random_state: int = 42, file="data/wq.csv"
) -> tuple[list[str, ...], float]:
    """returns the list of features and the accuracy score"""
 
    X_train, X_test, y_train, y_test = tts(get_wq_data(file=file), test_size=test_size, random_state=random_state)
 
    features = list(X_train.columns)

    score = DecisionTreeRegressor(random_state=random_state).fit(X_train[features], y_train).score(X_test[features], y_test)
    # raise NotImplementedError()
 
    return (features, score)

print("Coefficient of determination:",round(baseline_model(std_train_test_split)[1], 4))

Coefficient of determination: 0.6933


- Then, run the base model on our final dataset ```"good_vs_bad.pkl"```.

In [5]:
print("Coefficient of determination:",round(baseline_model(std_train_test_split, file="data/good_vs_bad.pkl")[1], 4))

Coefficient of determination: 0.8248
